# 56 — Operational Readiness

Sintetiza evidencia M6 y genera runbooks operacionales. `shadow_ready` y `production_promotion_authorized` son estados distintos: M6 puede demostrar operación segura en shadow sin autorizar promoción automática.

In [ ]:
import pandas as pd
from pathlib import Path

from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.production import load_production_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m5_gate = paths.docs / "decisions" / "M5_GATE.md"
assert m5_gate.exists() and "M5 ✅ CLOSED" in m5_gate.read_text(encoding="utf-8"), (
    "M5 no está formalmente cerrado. No iniciar M6."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M6_SPEC = load_production_spec(paths.configs / "production.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"
TEMP_DIR = paths.root / ".cache" / "duckdb"
SHADOW_INPUT = paths.root / "data" / "m5" / "fresh_oot_2026.parquet"

assert GOLD_SPEC.schema_version == M6_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M6_SPEC.core.benchmark_model_schema

{
    "m6_schema": M6_SPEC.schema_version,
    "model_version": M6_SPEC.core.model_version,
    "operating_mode": M6_SPEC.core.operating_mode,
    "scoring_mode": M6_SPEC.core.scoring_mode,
    "human_review_required": M6_SPEC.core.human_review_required,
    "production_promotion_authorized": M6_SPEC.deployment.allow_production_promotion,
}


In [ ]:
from argos.production.deployment_assets import ASSET_BOOTSTRAP_IMPLEMENTATION, ensure_deployment_assets
from argos.production.readiness import BUILD_IMPLEMENTATION, build_operational_readiness

print("asset bootstrap:", ASSET_BOOTSTRAP_IMPLEMENTATION)
print("readiness implementation:", BUILD_IMPLEMENTATION)
asset_status = ensure_deployment_assets(paths.root)
display(pd.DataFrame(asset_status))
assert all(item["exists"] for item in asset_status)

result = build_operational_readiness(
    root=paths.root, production_spec=M6_SPEC, reports_root=paths.reports, manifest_dir=MANIFEST_DIR
)
result.evidence


In [ ]:
result.validation

In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
failed = blocking.loc[~blocking["passed"], ["check_id", "category", "observed", "requirement"]]

print("56 ✅ PASS" if result.ready_for_next_notebook else "56 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
if not failed.empty:
    print("\nBlocking failures:")
    display(failed)
print("Operational runbook:", result.runbook_path)
print("Incident response:", result.incident_path)
print("Human review SOP:", result.human_review_path)
print("Retraining policy:", result.retraining_path)
print("Security & access:", result.security_path)
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook
